# Lognormal kappa sweep on RANDOMIZED starting capital (v3) — Colab T4

Third iteration of `colab_ks_wealth_lognormal_random_sweep.ipynb` (v2). The
economic design is unchanged: `kappa ~ LogNormal(mu, sigma)`, `mu = -sigma^2/2`
(mean fixed to 1), sigma from 0 (homogeneous) to 1 (wide spread), kappas drawn
as genuine i.i.d. samples, and a list of seeds swept at every sigma.

**What is new is underneath.** Agents no longer all start at the same capital.
The environment now draws per-agent starting capital `k_0 ~ U(3, 20)`,
independently per parallel env, redrawn at every episode reset — the
initialization the reference KS implementation uses. This sweep is the first
full-scale test of that change.

Those bounds are the reference's `U(10, 70)` rescaled to this repo's
calibration: the reference spans 0.25x–1.75x of *its* steady state
(`K* ~ 40` at `beta=0.99`), and `K* ~ 11.7` at the `beta=0.95` used here.

**A cell is exactly `ks_n200_top`** — `n_agents=200`, `num_envs=32`,
`total_timesteps=128000`, and the rest — with only the capital initialization
and the kappa vector changed. The full protocol is declared explicitly in
`config.yaml` and re-asserted before every cell trains, so it cannot drift.
(v2 used `n_agents=500` / `num_envs=8`, so v3 differs from v2 in population
as well as initialization.)

> **v3 cells are NOT comparable to v2's.** v2 stays on disk as the
> constant-`k_init` record.

Full design writeup: `runs/ks-wealth-lognormal-random-3/README.md`.

30 cells (6 sigmas x 5 seeds). **Time the first cell before assuming the rest
fit in one Colab session.**

In [ ]:
# Setup: clone or update the repo, install (idempotent -- safe to re-run).
%cd /content
![ -d jax-marl-bc ] || git clone https://github.com/danmonuni/jax-marl-bc.git
%cd jax-marl-bc
!git pull
!pip install -q -r requirements.txt && pip install -q -e . --no-deps

In [ ]:
# Sanity: a GPU runtime is attached (Runtime > Change runtime type > T4 GPU).
!nvidia-smi -L

In [ ]:
# Mount Drive BEFORE the run so results are saved as soon as they finish
# (a Colab disconnect then loses at most the run in progress, never a
# finished one).
import os, shutil
from google.colab import drive
drive.mount('/content/drive')

def save_results(name='ks-wealth-lognormal-random-3'):
    """Sync runs/<name>/results -> Drive (exact path, idempotent re-sync)."""
    src = f'runs/{name}/results'
    dst = f'/content/drive/MyDrive/jax-marl-bc-runs/{name}/results'
    assert os.path.exists(src), f"{src} missing - did the sweep finish?"
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"saved {src} -> {dst}")

## Verify the protocol before burning GPU hours

One tiny CPU cell that prints what will actually run. Two things to check:

1. the `k_0` line reads `U(3, 20) per agent, resampled per_episode` — if it
   says `(constant)`, the override did not take and the sweep would silently
   reproduce v2's initialization;
2. the printed protocol matches `ks_n200_top` (the smoke overrides below
   shrink `n_agents`/`num_envs`/`total_timesteps` only so it finishes in
   seconds — the real run uses 200 / 32 / 128000).

`verify_protocol()` also asserts this per cell and aborts on any mismatch.

In [ ]:
!python runs/ks-wealth-lognormal-random-3/sweep_lognormal_random_3.py device=cpu sim_steps=200 "sigmas=[0.5]" "seeds=[0]" \
    protocol.env.n_agents=16 protocol.train.total_timesteps=2000 \
    protocol.train.num_envs=4 protocol.train.num_minibatches=2 \
    out_dir=/tmp/v3_smoke 2>&1 | grep -E "k_0|economy|^  env\.|^  train\." 

## Sigma x seed sweep

Runs `runs/ks-wealth-lognormal-random-3/config.yaml` as-is: the `ks_n200_top` protocol
(`n_agents=200`, `num_envs=32`, `total_timesteps=128000`, ...) with
`k_0 ~ U(3,20)` resampled per episode, over
`sigmas=[0.0,0.2,0.4,0.6,0.8,1.0]` x `seeds=[0,1,2,3,4]`.

Dotlist overrides go after the script path; protocol entries are addressed
nested:

```
!python runs/ks-wealth-lognormal-random-3/sweep_lognormal_random_3.py device=cpu "sigmas=[0.0,0.5]" "seeds=[0,1]"
!python runs/ks-wealth-lognormal-random-3/sweep_lognormal_random_3.py protocol.train.num_envs=8 protocol.env.n_agents=500   # v2's shape
!python runs/ks-wealth-lognormal-random-3/sweep_lognormal_random_3.py k_init_dist=constant     # == ks_n200_top exactly, for a clean A/B
!python runs/ks-wealth-lognormal-random-3/sweep_lognormal_random_3.py "seeds=[0,1,2,3,4,5,6,7,8,9]"
```

That `k_init_dist=constant` form is the cleanest isolation of the change:
same protocol, same kappa design, same seeds, only the initialization
differs.

In [ ]:
!python runs/ks-wealth-lognormal-random-3/sweep_lognormal_random_3.py

In [ ]:
save_results('ks-wealth-lognormal-random-3')

## Results

In [ ]:
import pandas as pd
from IPython.display import Image, display

df = pd.read_csv('runs/ks-wealth-lognormal-random-3/results/results.csv')

# Confirm on the record that every cell ran the intended initialization.
print(df[['k_init_dist', 'k_init_low', 'k_init_high', 'k_init_resample']].drop_duplicates().to_string(index=False))

display(df[['sigma', 'seed', 'kappa_std', 'capital_gini', 'top_0.1_share', 'top_0.01_share', 'K_mean', 'euler_mean_abs']])
display(Image('runs/ks-wealth-lognormal-random-3/results/comparison.png'))

## Per-cell steady-state dashboards

One dashboard per `(sigma, seed)` cell: kappa profile, aggregate capital path
(with the `k_0` draw's support shaded, so you can see K converge to the same
steady state from anywhere in that band), aggregate consumption path, the
aggregate KS shock, the Lorenz curve, and the wealth histogram.

In [ ]:
for _, row in df.iterrows():
    sigma, seed = row['sigma'], int(row['seed'])
    print(f"sigma = {sigma:.2f}  seed = {seed}")
    display(Image(f'runs/ks-wealth-lognormal-random-3/results/figures/sigma_{sigma:.2f}_seed_{seed}_steady_state.png'))